# Chương 5: KG – RAG, giải thích từng hàm và chạy demo

    Corpus, embedding và generator đã có trong dự án. Run All chạy cục bộ,
    không cần API key. Phân biệt truy hồi bằng chứng với tính đúng của đáp án.

In [1]:
# Cell KG-01
import os, sys, json, time, inspect, copy
from pathlib import Path
for key in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ[key] = '2'
os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p/'src/tieuluan').exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
OUT = ROOT/'results/tieuluan/complete/notebooks'
OUT.mkdir(parents=True, exist_ok=True)
print('Project:', ROOT)

Project: E:\WINDOW\Project\intelligent-system-assignment-01


## 1. Đồ thị là hợp các bước chuẩn hóa → tạo nút → tạo cạnh → truy vấn

    `build_document_graph(chunks, domain)` nhận các đoạn đã có metadata và trả
    NetworkX DiGraph. Ví dụ dưới dùng hai dòng giả định để dễ kiểm tra; không
    coi các con số minh họa là báo cáo kinh doanh thật.

In [2]:
# Cell KG-02
from src.chapter04.knowledge import build_document_graph, safe_calculate
chunks = [{'id':'demo:row1','company':'DEMO','report':'DEMO:2023',
    'year':'2023','filename':'demo-report','kind':'table',
    'text':'Revenue 2022: 100 USD; 2023: 120 USD',
    'source_url':'https://example.org/demo'}]
graph = build_document_graph(chunks,'finance')
print('Nút:',list(graph.nodes))
print('Cạnh:',list(graph.edges(data=True)))
print('Nguồn cha:',list(graph.predecessors('passage:demo:row1')))
print(inspect.getsource(build_document_graph))

Nút: ['company:DEMO', 'report:DEMO:2023', 'passage:demo:row1']
Cạnh: [('company:DEMO', 'report:DEMO:2023', {'relation': 'PUBLISHED'}), ('report:DEMO:2023', 'passage:demo:row1', {'relation': 'HAS_PASSAGE'})]
Nguồn cha: ['report:DEMO:2023']
def build_document_graph(chunks,domain):
    graph=nx.DiGraph(domain=domain)
    for chunk in chunks:
        if domain=='finance':
            company='company:'+chunk['company'];report='report:'+chunk['report']
            graph.add_node(company,type='Company',ticker=chunk['company'])
            graph.add_node(report,type='Report',year=chunk['year'],filename=chunk['filename'],source_url=chunk['source_url'])
            graph.add_edge(company,report,relation='PUBLISHED')
        else:
            report='document:'+chunk['source_url']
            graph.add_node(report,type='Document',source_url=chunk['source_url'])
        node='passage:'+chunk['id']
        graph.add_node(node,type='TableRow' if chunk.get('kind')=='table' else 'Passage',chunk_id=ch

## 2. RRF: hợp thứ hạng thay vì cộng điểm khác thang đo

    `reciprocal_rank_fusion` nhận nhiều danh sách id và cộng 1/(60+rank).
    Ví dụ hai danh sách dưới giúp tự tính điểm của B. Hàm trả về id theo điểm
    giảm dần; bản thân RRF chưa tạo văn bản trả lời.

In [3]:
# Cell KG-03
from src.chapter04.retrieval import reciprocal_rank_fusion
rankings = [['A','B','C'],['B','D','A']]
print('Hạng hợp nhất:',reciprocal_rank_fusion(rankings,k=4))
print('Điểm B:',1/62+1/61)
print(inspect.getsource(reciprocal_rank_fusion))

Hạng hợp nhất: ['B', 'A', 'D', 'C']
Điểm B: 0.03252247488101534
def reciprocal_rank_fusion(rankings,k=5,c=60):
    scores=defaultdict(float)
    for ranking in rankings:
        for rank,cid in enumerate(ranking,1):scores[cid]+=1/(c+rank)
    return sorted(scores,key=lambda cid:(-scores[cid],cid))[:k]



## 3. Hợp hàm RAG: retrieve → build_prompt → generate → đóng gói nguồn

    Chạy truy hồi thật trước khi sinh. In id, locator và một phần văn bản để
    kiểm tra. `build_prompt` đặt câu hỏi trước ngữ cảnh. Kết quả được sinh giữ
    trạng thái generated_unverified: nguồn tồn tại chưa chứng minh đáp án đúng.

In [4]:
# Cell KG-04
from deployment.chapter04.app import rag
from src.chapter04.rag import build_prompt
engine = rag()
question = 'What are the symptoms of diabetes?'
evidence = engine.retrieve(question,'diabetes',k=3)
display(pd.DataFrame([{'id':e['id'],'text':e['text'][:180]}
                      for e in evidence]))
print(build_prompt(question,evidence)[:1400])
answer = engine.answer(question,'diabetes')
print('Answer:',answer['answer'])
print('Status:',answer['status'])
print('Sources:',answer['citations'])

,id,text
0,symptoms-causes:112,What clinical trials about symptoms and causes...
1,symptoms-causes:114,What have we learned about symptoms and causes...
2,symptoms-causes:107,Researchers are studying many aspects of sympt...


Answer using the context. If the answer is absent, say unknown.
Question: What are the symptoms of diabetes?
Context: What clinical trials about symptoms and causes of diabetes are looking for participants?

What have we learned about symptoms and causes of diabetes from NIDDK-funded research?

Researchers are studying many aspects of symptoms and causes of diabetes, such as
Answer:


Answer: NIDDK-funded research
Status: generated_unverified
Sources: [{'id': 'symptoms-causes:112', 'url': 'https://www.niddk.nih.gov/health-information/diabetes/overview/symptoms-causes', 'locator': 'symptoms-causes:112'}, {'id': 'symptoms-causes:114', 'url': 'https://www.niddk.nih.gov/health-information/diabetes/overview/symptoms-causes', 'locator': 'symptoms-causes:114'}, {'id': 'symptoms-causes:107', 'url': 'https://www.niddk.nih.gov/health-information/diabetes/overview/symptoms-causes', 'locator': 'symptoms-causes:107'}]


## 4. Tách phép tính ra công cụ

    `safe_calculate` nhận operation, hai values và units. Ví dụ growth từ
    100 lên 120 bằng 20%. Không dùng eval; không trích toán hạng tự động.
    Đơn vị không khớp hoặc chia cho không phải bị từ chối.

In [5]:
# Cell KG-05
print(safe_calculate('growth',[100,120],['USD','USD']))
for operation, values, units in [
    ('divide',[1,0],['USD','USD']),
    ('growth',[100,120],['USD','VND'])]:
    try:
        safe_calculate(operation,values,units)
    except ValueError as exc:
        print('Từ chối đúng:',exc)
    else:
        raise AssertionError('Đầu vào sai không bị chặn')

{'value': 20.0, 'formula': '(120.0 - 100.0) / 100.0 × 100%', 'unit': '%'}
Từ chối đúng: Division by zero
Từ chối đúng: Units must match


## 5. Đọc kết quả thực nghiệm đã lưu

    Các bảng này đọc từ JSON thí nghiệm gốc, không chạy lại benchmark trong
    notebook demo. So sánh global với oracle_report phải ghi đúng điều kiện.
    Explain: Hit@5, Recall@5, Complete@5 khác nhau thế nào?

In [6]:
# Cell KG-06
result = json.loads((ROOT/'results/chapter04/finance_rag_results.json')
                    .read_text(encoding='utf8'))
display(pd.DataFrame(result['retrieval'])[
    ['method','scope','hit_at_5','recall_at_5','complete_at_5']])
display(pd.DataFrame(result['generation']))
(OUT/'kg_demo.json').write_text(json.dumps({
    'question':question,'answer':answer['answer'],
    'status':answer['status'],
    'evidence_ids':[e['id'] for e in evidence]},
    ensure_ascii=False,indent=2),encoding='utf8')

,method,scope,hit_at_5,recall_at_5,complete_at_5
0,bm25_raw,global,0.519819,0.403380,0.305776
1,bm25_raw,oracle_report,0.809740,0.676948,0.552661
2,bm25_headers,global,0.619479,0.509377,0.408834
3,bm25_headers,oracle_report,0.894677,0.783794,0.673839
4,dense,global,0.621744,0.505015,0.395243
5,dense,oracle_report,0.913930,0.798022,0.671574
6,hybrid,global,0.682899,0.560717,0.448471
7,hybrid,oracle_report,0.926387,0.819974,0.705549


,mode,questions,numeric_accuracy,mean_seconds
0,gold_context_oracle,24,0.0,0.444281
1,hybrid_rag_oracle_report,24,0.0,0.559145
2,no_context,24,0.0,0.609651


232

## Giải thích từng hàm khi bảo vệ

    build_document_graph: tạo schema và cạnh có nguồn; retrieve: chọn đoạn;
    reciprocal_rank_fusion: hợp hạng; build_prompt: ghép câu hỏi và ngữ cảnh;
    generate: sinh văn bản; answer: đóng gói nguồn và trạng thái;
    safe_calculate: tính số theo toán hạng cấu trúc. RAG không bảo đảm hết ảo giác.